In [ ]:
# Install required packages (auto-skipped if already installed)
import importlib
if importlib.util.find_spec('qiskit') is None:
    !pip install -q qiskit qiskit-aer qiskit-ibm-runtime pylatexenc samplomatic
else:
    print("\u2713 Packages already installed")

# To run on real quantum hardware, uncomment and fill in your credentials:
# from qiskit_ibm_runtime import QiskitRuntimeService
# QiskitRuntimeService.save_account(
#     token="<your-api-key>",
#     instance="<your-crn>",
#     overwrite=True
# )

# מדריך מהיר ל-Executor

<Accordion>
<AccordionItem title="גרסאות חבילות">

הקוד בדף זה פותח תוך שימוש בדרישות הבאות.
אנחנו ממליצים להשתמש בגרסאות אלה או חדשות יותר.

```
qiskit[all]~=2.4.0
qiskit-ibm-runtime~=0.46.1
samplomatic~=0.18.0
```
</AccordionItem>
</Accordion>

בדומה ל-primitive של [Sampler](/guides/get-started-with-sampler), Executor דוגם רגיסטרי פלט מהרצות מעגלים קוונטיים, אך אין לו דיכוי שגיאות או הפחתת שגיאות מובנים. במקום זאת, הוא חלק מ[מודל ההרצה המכוונת](/guides/directed-execution-model) שמספק את הרכיבים ללכידת כוונות עיצוב בצד הלקוח, ומעביר את הייצור היקר של וריאנטי מעגלים לצד השרת. Executor עוקב אחר ההנחיות שניתנו ב-annotations של המעגל ובאפשרויות, מייצר ומקשר ערכי פרמטרים, מריץ את המעגלים המחוברים על החומרה, ומחזיר את תוצאות ההרצה ומטא-הנתונים. הוא לא מקבל עבורך שום החלטות מרומזות ונותן לך שליטה ושקיפות מלאה.

> **Note:** לחבילת Qiskit עדיין אין מחלקת בסיס עבור ה-primitive של Executor.

## לפני שמתחילים
חלק מדוגמאות הקוד בדף זה משתמשות ב-`samplex`, שהוא חלק מחבילת Samplomatic. לכן, לפני הרצת בלוקי קוד אלה, חייבים להתקין את Samplomatic, כפי שמוצג בבלוק הקוד הבא. למידע נוסף, ראו את [תיעוד Samplomatic](https://qiskit.github.io/samplomatic).

In [1]:
from qiskit_ibm_runtime import QiskitRuntimeService, Executor
from qiskit_ibm_runtime.quantum_program import QuantumProgram
from qiskit.circuit import QuantumCircuit
from qiskit.transpiler import generate_preset_pass_manager
from samplomatic.transpiler import generate_boxing_pass_manager
from samplomatic import build

# Initialize the service and choose a backend
service = QiskitRuntimeService()
backend = service.least_busy(operational=True, simulator=False)

In [2]:
print(backend)

<IBMBackend('ibm_fez')>


### 2. Create and transpile a circuit

You need at least one circuit to use the Executor primitive.  It can optionally have parameters.

In [3]:
# Generate the circuit
circuit = QuantumCircuit(2)
circuit.h(0)
circuit.h(1)
circuit.cz(0, 1)
circuit.h(1)

# Using `measure_all` automatically creates the necessary
# classical registers.
circuit.measure_all()

The circuit needs to be transformed to only use instructions supported by the QPU (referred to as *instruction set architecture (ISA)* circuits). Use the transpiler to do this.

In [4]:
# Transpile the circuit
preset_pass_manager = generate_preset_pass_manager(
    backend=backend, optimization_level=0
)
isa_circuit = preset_pass_manager.run(circuit)

### 2. יצירה ו-transpile של Circuit
צריך לפחות Circuit אחד כדי להשתמש ב-primitive של Executor. הוא יכול לכלול פרמטרים אופציונלית.

In [5]:
# Initialize an empty program
program = QuantumProgram(shots=25)

# Append the circuit to the program
program.append_circuit_item(isa_circuit)

ה-Circuit צריך להיות מוּמר כך שישתמש רק בהוראות הנתמכות על ידי ה-QPU (המכונות מעגלי *instruction set architecture (ISA)*). השתמש ב-Transpiler לשם כך.

In [6]:
# Generate a boxing pass manager to group gates
# and measurements into boxes and add
# a`Twirl` annotation.
boxes_pm = generate_boxing_pass_manager(
    # Add gate twirling
    enable_gates=True,
    # Add measurement twirling
    enable_measures=True,
)

boxed_circuit = boxes_pm.run(isa_circuit)
boxed_circuit.draw("mpl", idle_wires=False)

<Image src="../docs/images/guides/get-started-with-executor/extracted-outputs/245a4574-3ce9-4f77-98c8-af32cde8ac01-0.svg" alt="Output of the previous code cell" />

### 3. אתחול `QuantumProgram`
אתחל `QuantumProgram` עם עומס העבודה שלך. `QuantumProgram` מורכב מ-`QuantumProgramItems`. בדרך כלל, כל פריט מורכב ממעגל, סדרת ערכי פרמטרים, ואפשרית `samplex` לאקראיות תוכן המעגל. לפרטים מלאים, ראו [קלטים ופלטים של Executor](/guides/executor-input-output).

התא הבא מאתחל `QuantumProgram` ומציין לבצע 25 shots. לאחר מכן הוא מוסיף את מעגל ה-transpile היעד.

In [7]:
# Build the template circuit and the samplex
template_circuit, samplex = build(boxed_circuit)

# Append the template circuit and samplex as a `samplex_item`
program.append_samplex_item(
    template_circuit,
    samplex=samplex,
    shape=(num_randomizations := 20,),
)

### 4. אופציונלי: קיבוץ שערים ומדידות ל-boxes מסומנים
קיבוץ הוראות ל-boxes וסימונן הוא הדרך העיקרית לציין את הכוונה שלך. בדוגמה הבאה, אנחנו משתמשים ב-`generate_boxing_pass_manager` ובפרמטרי ה-twirling שלו כדי לקבץ שערים דו-Qubit ומדידות ל-boxes ולהוסיף annotation של twirling.

In [8]:
# Initialize an Executor with the default options
executor = Executor(mode=backend)

# Submit the job
job = executor.run(program)
job

<RuntimeJobV2('d8286580bvlc73d1vmsg', 'executor')>

In [9]:
# Retrieve the result
result = job.result()

### 6. הפעלת Executor וקבלת תוצאות
הרץ את `QuantumProgram` על Backend של IBM&reg; על ידי שימוש ב-primitive של `Executor` עם אפשרויות ברירת מחדל. ראו [אפשרויות Executor](/guides/executor-options) כדי ללמוד על האפשרויות הזמינות.